# Quantization intro

This notebook focuses on post-training quantization methods for Google Gemma 3 model with 270m parameters.

## Installation

In [1]:
%pip install -U transformers accelerate datasets huggingface_hub llmcompressor hqq bitsandbytes

  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)


In [2]:
import torch
import bitsandbytes as bnb
import gc
import numpy as np
import matplotlib.pyplot as plt
import json
# import dotenv
# import os
import pandas as pd

from pathlib import Path
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier, QuantizationModifier
from compressed_tensors.offload import dispatch_model
from llmcompressor.modifiers.transform.awq import AWQModifier
from hqq.core.quantize import BaseQuantizeConfig
from hqq.models.hf.base import AutoHQQHFModel

# from scripts.load_prompts import load_prompts_from_json
# dotenv.load_dotenv()

In [3]:
login(token=getpass("Hugging Face token: "), add_to_git_credential=False)


Hugging Face token: ··········


In [4]:
# device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
NUM_CALIBRATION_SAMPLES = 64
MAX_SEQ_LENGTH = 256
VALIDATION_PROMPTS_PATH = "./intro_prompts.json"
# cwd = Path.cwd()
# PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
# VALIDATION_PROMPTS_PATH = (PROJECT_ROOT / "data" / "raw" / "intro_prompts.json")
# assert VALIDATION_PROMPTS_PATH.exists(), VALIDATION_PROMPTS_PATH

### Helper functions

In [21]:
def clear_memory():
    """
    Helper function to run garbage collection
    and release unused CUDA/MPS cached memory.
    """
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def describe_model(model):
    """
    Helper function to retrieve information
    about model name and layers.
    """
    return type(model).__name__, model.model.layers

def model_size_mb(model):
    """
    Helper function to retrieve model size in MiB.
    """
    return model.get_memory_footprint() / 1_000_000

def load_baseline_model():
    """
    Helper function to load baseline model
    as a template for quantization methods.
    """
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device).eval()

def tokenize_recieve(model, prompt):
    """
    Helper function to tokenize the prompt
    and recieve an answer.
    """
    message = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(message, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer

def load_prompts_from_json(filename: str) -> tuple:
    """
    Helper function to load the prompts from the JSON file.

    :param filename: str, path to the JSON file.
    :return: tuple, prompts and reference_answers
    """
    with open(filename) as f:
        data = json.load(f)
        prompts = [r["question"] for r in data]
        reference_answers = [r["answer"] for r in data]

        return prompts, reference_answers


def evaluate_model(model, tokenizer):
    """
    Helper function to evaluate model's loss
    """
    model.eval()
    device = next(model.parameters()).device

    validation_data, _ = load_prompts_from_json(VALIDATION_PROMPTS_PATH)

    total_loss, total_tokens = 0.0, 0

    for prompt in validation_data:
        inputs = tokenizer(prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count

    return { "loss": total_loss / total_tokens}

def model_results(baseline_model, baseline_model_loss, baseline_model_size, quantized_model, method_name):
    """
    Helper function for checking and comparing model's results.
    """
    evaluation_results_quantization = evaluate_model(quantized_model, tokenizer)
    loss_q = evaluation_results_quantization["loss"]

    example_prompt = "Explain quantization of Large Language Models in 1 sentence."

    print(f"Baseline loss: {baseline_model_loss:.4f}")
    print(f"{method_name} loss: {loss_q:.4f}")

    print(f"\nBaseline model size (MB): {baseline_model_size:.3f}")
    print(f"Quantized {method_name} model size (MB): {model_size_mb(quantized_model):.3f}")

    print(f"\nPrompt: {example_prompt}")
    print(f"\nOutput Baseline: {tokenize_recieve(baseline_model, example_prompt).strip("\n")}")
    print(f"Output {method_name}: {tokenize_recieve(quantized_model, example_prompt).strip("\n")}")


## Baseline model

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(device)
model.eval()

model_type, baseline_model_descr = describe_model(model)

print(f"Model loaded on: {model.device}")
print(model_type)
print(baseline_model_descr)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cuda:0
Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layerno

In [7]:
res = tokenize_recieve(model, "Explain machine learning in 2 simple sentences.")
print(res)

Machine learning is a type of computer science that uses algorithms to learn from data without being explicitly programmed.



In [25]:
baseline_model = load_baseline_model()
evaluation_results_baseline = evaluate_model(baseline_model, tokenizer)
baseline_model_loss = evaluation_results_baseline["loss"]
baseline_size = model_size_mb(baseline_model)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

### Calibration Dataset preparation

In [24]:
c4_dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

calibration_texts = [row["text"] for row in c4_dataset.take(NUM_CALIBRATION_SAMPLES)]
encoded = tokenizer(calibration_texts, truncation=True, max_length = MAX_SEQ_LENGTH, padding=False, add_special_tokens=False)
calibration_dataset = Dataset.from_dict(dict(encoded))

print("Calibration samples:", len(calibration_dataset))
print(calibration_dataset)

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Calibration samples: 64
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 64
})


## Linear symmetric and asymmetric quantization

In [10]:
def symmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Symmetric Quantization per tensor.
    Zero point is fixed at 0.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor], downcast tensor, scale
    """
    max_original = x.abs().max()
    max_int = torch.iinfo(dtype).max
    s = max_original / max_int
    s = torch.where(s == 0, torch.ones_like(s), s)
    q = torch.round(x / s).clamp(-max_int, max_int).to(dtype)

    return q, s

def asymmetric_quantize(x: torch.Tensor, dtype=torch.int8) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Function implements Linear Asymmetric Quantization per tensor.

    :param x: torch.Tensor, single tensor
    :param dtype: torch.dtype, type to downcast tensor to
    :return: tuple[torch.Tensor, torch.Tensor, torch.Tensor], downcast tensor, scale, zero point
    """
    min_original, max_original = x.min().clamp(max=0), x.max().clamp(min=0)
    min_int, max_int = torch.iinfo(dtype).min, torch.iinfo(dtype).max

    s = (max_original - min_original) / (max_int - min_int)
    s = torch.where(s == 0, torch.ones_like(s), s)

    z = torch.round(min_int - min_original / s).clamp(min_int, max_int).to(dtype)
    q = (torch.round(x / s) + z.float()).clamp(min_int, max_int).to(dtype)

    return q, s, z

def dequantize(q: torch.Tensor, s: torch.Tensor, z: int | torch.Tensor = 0) -> torch.Tensor:
    """
    Function implements Linear Dequantization per tensor.

    :param q: torch.Tensor, quantized tensor
    :param s: torch.Tensor, scale
    :param z: torch.Tensor, zero point
    :return: torch.Tensor, dequantized tensor
    """
    return s.float() * (q.float() - torch.as_tensor(z, dtype=torch.float32, device=q.device))


In [11]:
@torch.no_grad()
def linear_weights_quantization(model, method="symmetric") -> list:
    """
    Function implements Linear Quantization on Gemma 3 model weights.

    :param model: Gemma 3 model
    :param method: str, symmetric or asymmetric
    :return: list, quantization results
    """
    results = []
    seen_params = set()
    model_type, model_layers = describe_model(model)

    for name, module in model_layers.named_modules():
        if not (isinstance(module, torch.nn.Linear)):
            continue

        param = module.weight

        if id(param) in seen_params:
            continue

        seen_params.add(id(param))
        weights = param.detach().to(device="cpu", dtype=DTYPE).clone()

        if method == "symmetric":
            q, s = symmetric_quantize(weights)
            restored = dequantize(q, s)
            zero_point = 0
        else:
            q, s, z = asymmetric_quantize(weights)
            restored = dequantize(q, s, z)
            zero_point = z.item()

        results.append({
            "layer": name,
            "shape": tuple(weights.shape),
            "num_weights": weights.numel(),
            "scale": s.item(),
            "zero_point": zero_point
        })

        param.copy_(restored.to(device=param.device, dtype=param.dtype))
        del weights, q, restored, s

    return results

In [27]:
symmetric_model = load_baseline_model()

results_sym = linear_weights_quantization(symmetric_model, method="symmetric",)
prompt_results_symmetric = model_results(baseline_model, baseline_model_loss, baseline_size, symmetric_model, "Linear Symmetric")

del symmetric_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Baseline loss: 3.6954
Linear Symmetric loss: 3.9431

Baseline model size (MB): 1072.395
Quantized Linear Symmetric model size (MB): 1072.395

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Symmetric: Large Language Models (LLMs) are designed to process and understand natural language, achieving a state-of-the-art performance in various tasks.


In [28]:
asymmetric_model = load_baseline_model()

results_asym = linear_weights_quantization(asymmetric_model, method="asymmetric",)
prompt_results_asymmetric = model_results(baseline_model, baseline_model_loss, baseline_size, asymmetric_model, "Linear Asymmetric")

del asymmetric_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Baseline loss: 3.6954
Linear Asymmetric loss: 3.7900

Baseline model size (MB): 1072.395
Quantized Linear Asymmetric model size (MB): 1072.395

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output Linear Asymmetric: Quantization reduces the model's size and computational cost, making it suitable for lower-resource environments.


## NF4 model - bitsandbytes

In [29]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)

nf4_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization_config, device_map={"": 0}, dtype=DTYPE).eval()
prompt_results_nf4 = model_results(baseline_model, baseline_model_loss, baseline_size, nf4_model, "NF4")

del nf4_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Baseline loss: 3.6954
NF4 loss: 4.2694

Baseline model size (MB): 1072.395
Quantized NF4 model size (MB): 721.449

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output NF4: Quantization of Large Language Models (LLMs) aims to reduce the size and complexity of the model by lowering the number of parameters, typically to fractions of the original size. This reduces the model's computational cost, memory requirements, and training time, leading to improved performance on specific, often less-studied, tasks.


## GPTQ

In [30]:
gptq_model = load_baseline_model()
gptq_recipe = GPTQModifier(targets=["Linear"], scheme="W4A16", ignore=["lm_head"], dampening_frac=0.1, actorder=None,)

oneshot(model=gptq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=gptq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False,)

dispatch_model(gptq_model)
prompt_results_gptq = model_results(baseline_model, baseline_model_loss, baseline_size, gptq_model, "GPTQ")

del gptq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

2026-10-23T23:00:31.3414 | reset | INFO - Compression lifecycle reset
2026-10-23T23:00:31.3581 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-23T23:00:31.3918 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 1146.29it/s]

2026-10-23T23:00:31.5364 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-10-23T23:00:31.5404 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(2/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 162.67it/s]

...

2026-10-23T23:00:52.6711 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.q_proj']
2026-10-23T23:00:52.6813 | GPTQ | METRIC - time 0.01s
2026-10-23T23:00:52.6820 | GPTQ | METRIC - error 3472.35
2026-10-23T23:00:52.6830 | GPTQ | METRIC - Accelerator 0 | usage: 41.25% | total memory: 15.6 Gb
2026-10-23T23:00:52.6853 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.self_attn.k_proj', 'model.layers.17.self_attn.v_proj']
2026-10-23T23:00:52.7005 | GPTQ | METRIC - time 0.01s
2026-10-23T23:00:52.7023 | GPTQ | METRIC - error 1046.26
2026-10-23T23:00:52.7042 | GPTQ | METRIC - Accelerator 0 | usage: 41.25% | total memory: 15.6 Gb
2026-10-23T23:00:52.7048 | GPTQ | METRIC - time 0.02s


2026-10-23T23:00:52.7058 | GPTQ | METRIC - error 1121.16
2026-10-23T23:00:52.7069 | GPTQ | METRIC - Accelerator 0 | usage: 41.25% | total memory: 15.6 Gb
2026-10-23T23:00:52.7133 | compress_module_list | INFO - Quantizing 1 module(s): ['model.layers.17.self_attn.o_proj']
2026-10-23T23:00:52.7356 | GPTQ | METRIC - time 0.02s
2026-10-23T23:00:52.7364 | GPTQ | METRIC - error 244.35
2026-10-23T23:00:52.7374 | GPTQ | METRIC - Accelerator 0 | usage: 41.25% | total memory: 15.6 Gb
2026-10-23T23:00:52.7401 | compress_module_list | INFO - Quantizing 2 module(s): ['model.layers.17.mlp.gate_proj', 'model.layers.17.mlp.up_proj']
2026-10-23T23:00:52.7533 | GPTQ | METRIC - time 0.01s
2026-10-23T23:00:52.7554 | GPTQ | METRIC - error 269.93
2026-10-23T23:00:52.7567 | GPTQ | METRIC - Accelerator 0 | usage: 41.25% | total memory: 15.6 Gb
2026-10-23T23:00:52.7574 | GPTQ | METRIC - time 0.02s
2026-10-23T23:00:52.7582 | GPTQ | METRIC - error 275.79
2026-10-23T23:00:52.7601 | GPTQ | METRIC - Accelerator 0 |

(19/19): Propagating: 100%|██████████| 64/64 [00:00<00:00, 178.48it/s]

2026-10-23T23:00:53.1858 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention
2026-10-23T23:00:53.1996 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


Baseline loss: 3.6954
GPTQ loss: 4.9118

Baseline model size (MB): 1072.395
Quantized GPTQ model size (MB): 1076.312

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output GPTQ: Large language models (LLMs) are designed to be quantized to represent a range of values, allowing for efficient and accurate calculations.

Here are some examples:

*   **Low-precision:**  LLMs are often used in low-precision formats like FP16 or BF455.
*   **Mid-precision:**  LLMs are often used in mid-precision formats like FP34 or FP16.
*   **High-precision:**  LLMs


## AWQ

In [31]:
awq_model = load_baseline_model()
awq_recipe = [AWQModifier(), QuantizationModifier(targets=["Linear"], scheme="W4A16_ASYM", ignore=["lm_head"])]

oneshot(model=awq_model, tokenizer=tokenizer, dataset=calibration_dataset, recipe=awq_recipe,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=len(calibration_dataset),
        batch_size=1,
        shuffle_calibration_samples=False)

dispatch_model(awq_model)
prompt_results_awq = model_results(baseline_model, baseline_model_loss, baseline_size, awq_model, "AWQ")

del awq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

2026-10-23T23:01:31.6547 | reset | INFO - Compression lifecycle reset
2026-10-23T23:01:31.6594 | norm_calibration_context | INFO - Found 109 offset-norm modules to convert
2026-10-23T23:01:31.6719 | from_modifiers | INFO - Creating recipe from modifiers
2026-10-23T23:01:31.6726 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...


Applying quantization config: 100%|██████████| 126/126 [00:00<00:00, 5678.39it/s]

2026-10-23T23:01:31.7051 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-10-23T23:01:31.7058 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`
2026-10-23T23:01:31.7976 | _set_resolved_mappings | WARNING - 18 mappings were skipped due to incompatible shapes. Set logging to DEBUG to view full list.



(1/19): Calibrating: 100%|██████████| 64/64 [00:00<00:00, 526.28it/s]
Smoothing: 0it [00:00, ?it/s]
Smoothing:   0%|          | 0/2 [00:00<?, ?it/s]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=1.134e-02]
Grid search for model.layers.0.pre_feedforward_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=9.846e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 17.28it/s, best_error=9.846e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 17.28it/s, best_error=9.201e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 17.28it/s, best_error=9.201e-03]
Grid search for model.layers.0.pre_feedforward_layernorm:  10%|█         | 2/20 [00:00<00:01, 17.28it/s, best_error=9.086e-03]
Grid se

2026-10-23T23:02:08.1202 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`
2026-10-23T23:02:08.2130 | norm_calibration_context | INFO - Restoring 109 norm modules to offset convention


2026-10-23T23:02:08.2268 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
Baseline loss: 3.6954
AWQ loss: 4.4147

Baseline model size (MB): 1072.395
Quantized AWQ model size (MB): 1076.312

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output AWQ: **Quantization** is the process of lowering the precision of a Large Language Model (LLM) in a way that it can only represent a certain number of tokens (variables) in a fixed size. This allows the model to perform tasks that require a large amount of computation, such as reasoning, solving complex problems, and generating creative text.


## HQQ

In [32]:
hqq_model = load_baseline_model()
quant_config = BaseQuantizeConfig(nbits=8, group_size=64)
AutoHQQHFModel.quantize_model(hqq_model, quant_config=quant_config, compute_dtype=DTYPE, device="cuda:0",)
prompt_results_hqq = model_results(baseline_model, baseline_model_loss, baseline_size, hqq_model, "HQQ")

del hqq_model
clear_memory()

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

100%|██████████| 127/127 [00:00<00:00, 266.44it/s]


Baseline loss: 3.6954
HQQ loss: 3.7028

Baseline model size (MB): 1072.395
Quantized HQQ model size (MB): 771.585

Prompt: Explain quantization of Large Language Models in 1 sentence.

Output Baseline: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
Output HQQ: Quantization reduces the size of a Large Language Model (LLM) while maintaining its performance, allowing for faster inference and lower computational costs.
